# Applying Spectral Derivative Pigment (SDP) Phytoplankton Community Composition Algorithm to OCI data

**Author(s):** Anna Windle (NASA, SSAI), Max Danenhower (Bowdoin College), Sasha Kramer (Boston University)

Last updated: June 25, 2026

<div class="alert alert-success" role="alert">

The following notebooks are **prerequisites** for this tutorial.

- Learn with OCI: [Data Access][oci-data-access]

</div>

<div class="alert alert-info" role="alert">

An [Earthdata Login][edl] account is required to access data from the NASA Earthdata system, including NASA ocean color data.

</div>

[edl]: https://urs.earthdata.nasa.gov/
[oci-data-access]: /notebooks/oci_data_access/

## Summary

This notebook applies the inversion algorithm described in [Kramer et al., 2022][Kramer-et-al] to estimate phytoplankton pigment concentrations from PACE OCI Rrs data. This algorithm, called Spectral Derivative Pigment (SDP), is currently being implemented in OBPG's OCSSW software. This work was originally [developed in MatLab](https://github.com/sashajane19/Rrs_pigments) by Sasha Kramer and subsequently [translated to Python](https://github.com/max-danenhower/rrs-SDP-pigments/) by Max Danenhower. This tutorial demonstrates how to apply the Python SDP algorithm to Level-2 (L2) and Level-3 Mapped (L3M) PACE OCI data.

[Kramer-et-al]: https://doi.org/10.1016/j.rse.2021.112879

## Learning Objectives

At the end of this notebook you will know:

- How to use a packaged Python project
- How to run the SDP Algorithm on PACE OCI L2 and L3 data

## 1. Setup

The SDP Python code has been packaged to allow it to be easily installed, imported, and reused.
While the package is not on PyPI or conda-forge, it can be installed directly from the [source repository][sdp] on GitHub.
If you have followed the setup instructions, then GPig is available to import along with the other packages needed for this notebook.

[sdp]: https://github.com/max-danenhower/rrs-SDP-pigments/

In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import earthaccess
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

crs = ccrs.PlateCarree()

In [2]:
pip install scikit-learn openpyxl ray

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))
import sdp

In [4]:
import sdp
print(sdp.__file__)

/home/jovyan/rrs-SDP-pigments/src/sdp/__init__.py


In [5]:
from sdp import rrsModelTrain

Set (and persist to your home directory on the host, if needed) your Earthdata Login credentials.

In [6]:
auth = earthaccess.login()

In [7]:
results = earthaccess.search_data(
    short_name=["PACE_OCI_L2_AOP"],
    temporal= ("2026-03-03", "2026-03-03"),
    granule_name='*20260303T183722*')
results

[Collection: {'Version': '3.2', 'ShortName': 'PACE_OCI_L2_AOP'}
 Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 1.3079, 'Longitude': -73.34805}, {'Latitude': 19.17547, 'Longitude': -76.59283}, {'Latitude': 14.0328, 'Longitude': -101.25039}, {'Latitude': -3.72917, 'Longitude': -97.05367}, {'Latitude': 1.3079, 'Longitude': -73.34805}]}}]}}}
 Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2026-03-03T18:37:22Z', 'EndingDateTime': '2026-03-03T18:42:21Z'}}
 Size(MB): 437.3645133972168
 Data: ['https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20260303T183722.L2.OC_AOP.V3_2.nc']]

In [8]:
paths = earthaccess.open(results)
paths

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

[<File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20260303T183722.L2.OC_AOP.V3_2.nc>]

In [9]:
paths[0]

<File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20260303T183722.L2.OC_AOP.V3_2.nc>

## 2. Run SDP on L2 Data

In [10]:
from sdp.core import sdp_from_pace

In [11]:
def load_data(tspan):
    '''
    Downloads one L2 PACE apparent optical properties (AOP) file that intersects the coordinate box passed in, as well as 
    temperature and salinity files. Data files are saved to local folders named 'L2_rrs_data', 'sal_data', and 'temp_data'.

    Parameters:
    -----------
    tspan : tuple of str
        A tuple containing two strings both with format 'YYYY-MM-DD'. The first date in the tuple must predate the second date in the tuple.
    bbox : tuple of floats or ints
        A tuple representing spatial bounds in the form (lower_left_lon, lower_left_lat, upper_right_lon, upper_right_lat).

    Returns:
    --------
    list
        A list of file paths to a PACE L2 AOP files intersecting the passed in bounding box.
    string
        A single file path to a salinity file.
    string
        A single file path to a temperature file.
    '''

    #L2_results = earthaccess.search_data(
    #    short_name='PACE_OCI_L2_AOP',
    #    bounding_box=bbox,
    #    temporal=tspan
    #)
    #if (len(L2_results) > 0):
    #    L2_paths = earthaccess.download(L2_results, 'L2_rrs_data')
    #else:
    #    L2_paths = []
     #   print('No L2 AOP data found')

    sal_results = earthaccess.search_data(
        short_name='SMAP_JPL_L3_SSS_CAP_8DAY-RUNNINGMEAN_V5',
        temporal=tspan,
        count=1
    )
    if (len(sal_results) > 0):
        sal_paths = earthaccess.download(sal_results, 'sal_data')
    else:
        sal_paths = []
        print('No salinity data found')

    temp_results = earthaccess.search_data(
        short_name='MUR-JPL-L4-GLOB-v4.1',
        temporal=tspan,
        count=1
    )
    if (len(temp_results) > 0):
        temp_paths = earthaccess.download(temp_results, 'temp_data')
    else:
        temp_paths = []
        print('No temperature data found')

    return sal_paths[0], temp_paths[0]

load_data(tspan = ("2026-03-03", "2026-03-03"))

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

(PosixPath('sal_data/SMAP_L3_SSS_20260227_8DAYS_V5.0.nc'),
 PosixPath('temp_data/20260303090000-JPL-L4_GHRSST-SSTfnd-MUR-GLOB-v02.0-fv04.1.nc'))

In [12]:
output_file = paths[0].full_name.split("/")[-1]
output_file = output_file.replace(".nc", "_SDP_pigments.nc")
output_file

'PACE_OCI.20260303T183722.L2.OC_AOP.V3_2_SDP_pigments.nc'

In [ ]:
import ray
ray.init(include_dashboard=False)
print(ray.available_resources())

In [ ]:
%%time

sdp_from_pace(paths[0], output_file, 
             sss_file='/home/jovyan/rrs-SDP-pigments/sal_data/SMAP_L3_SSS_20260427_8DAYS_V5.0.nc',
             sst_file='/home/jovyan/rrs-SDP-pigments/temp_data/20260501090000-JPL-L4_GHRSST-SSTfnd-MUR-GLOB-v02.0-fv04.1.nc')

In [ ]:
dat = xr.open_dataset('sdp_pigments-PACE_OCI.20260501T000450.L2.OC_AOP.V3_1.NRT_SDP_pigments.nc')
dat

In [ ]:
dat.chla.plot(vmax=1)

In [ ]:
dat.chlb.mean()